<a href="https://colab.research.google.com/github/Keerthan-AC-583/BIS-Lab/blob/main/1BM24CS133_PSO_5G_network_allocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# ============================================================
# PSO-Based 5G Resource Allocation
# Objective: Maximize total network throughput
# ============================================================

# -----------------------------
# 1. Network Parameters
# -----------------------------

N = 3                  # Number of users
P_max = 10.0           # Maximum total transmission power (W)
B_max = 10.0           # Total bandwidth (MHz)

# Channel gains for 3 users
channel_gain = np.array([1.0, 0.8, 0.5])

# Noise power spectral density
N0 = 1e-9

# Convert MHz to Hz
B_max_Hz = B_max * 1e6


# -----------------------------
# 2. PSO Parameters
# -----------------------------

num_particles = 30
max_iterations = 100

w = 0.7       # Inertia weight
c1 = 1.5      # Cognitive coefficient
c2 = 1.5      # Social coefficient


# ============================================================
# 3. Fitness Function
# ============================================================

def calculate_throughput(power, bandwidth):
    """
    Calculate total network throughput.

    power     : Power allocation for each user (W)
    bandwidth : Bandwidth allocation for each user (Hz)

    Returns:
        Total throughput in Mbps
    """

    total_rate = 0

    for i in range(N):

        if bandwidth[i] > 0:

            # Shannon capacity formula
            snr = (power[i] * channel_gain[i]) / (N0 * bandwidth[i])

            rate = bandwidth[i] * np.log2(1 + snr)

            total_rate += rate

    # Convert bits/sec to Mbps
    return total_rate / 1e6


# ============================================================
# 4. Generate a Valid Particle
# ============================================================

def generate_particle():

    # Random power allocation
    power = np.random.rand(N)

    # Normalize so total power = P_max
    power = (power / np.sum(power)) * P_max

    # Random bandwidth allocation
    bandwidth = np.random.rand(N)

    # Normalize so total bandwidth = B_max
    bandwidth = (bandwidth / np.sum(bandwidth)) * B_max_Hz

    return np.concatenate([power, bandwidth])


# ============================================================
# 5. Initialize Particles
# ============================================================

particles = np.array([
    generate_particle()
    for _ in range(num_particles)
])

# Initialize velocities
velocities = np.random.uniform(
    -1, 1,
    size=(num_particles, 2 * N)
)


# ============================================================
# 6. Personal Best
# ============================================================

personal_best = particles.copy()

personal_best_fitness = np.array([
    calculate_throughput(
        particle[:N],
        particle[N:]
    )
    for particle in particles
])


# ============================================================
# 7. Global Best
# ============================================================

best_index = np.argmax(personal_best_fitness)

global_best = personal_best[best_index].copy()

global_best_fitness = personal_best_fitness[best_index]


# ============================================================
# 8. PSO Main Loop
# ============================================================

for iteration in range(max_iterations):

    for i in range(num_particles):

        # Random values
        r1 = np.random.rand(2 * N)
        r2 = np.random.rand(2 * N)

        # --------------------------------
        # Update velocity
        # --------------------------------

        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (personal_best[i] - particles[i])
            + c2 * r2 * (global_best - particles[i])
        )

        # --------------------------------
        # Update position
        # --------------------------------

        particles[i] = particles[i] + velocities[i]

        # --------------------------------
        # Separate power and bandwidth
        # --------------------------------

        power = particles[i][:N]
        bandwidth = particles[i][N:]

        # --------------------------------
        # Make values non-negative
        # --------------------------------

        power = np.maximum(power, 0)
        bandwidth = np.maximum(bandwidth, 0)

        # --------------------------------
        # Normalize power
        # --------------------------------

        if np.sum(power) == 0:
            power = np.ones(N) * (P_max / N)
        else:
            power = (power / np.sum(power)) * P_max

        # --------------------------------
        # Normalize bandwidth
        # --------------------------------

        if np.sum(bandwidth) == 0:
            bandwidth = np.ones(N) * (B_max_Hz / N)
        else:
            bandwidth = (bandwidth / np.sum(bandwidth)) * B_max_Hz

        # Update particle
        particles[i] = np.concatenate([
            power,
            bandwidth
        ])

        # --------------------------------
        # Calculate fitness
        # --------------------------------

        fitness = calculate_throughput(
            power,
            bandwidth
        )

        # --------------------------------
        # Update Personal Best
        # --------------------------------

        if fitness > personal_best_fitness[i]:

            personal_best[i] = particles[i].copy()

            personal_best_fitness[i] = fitness

    # --------------------------------
    # Update Global Best
    # --------------------------------

    best_index = np.argmax(personal_best_fitness)

    if personal_best_fitness[best_index] > global_best_fitness:

        global_best = personal_best[best_index].copy()

        global_best_fitness = personal_best_fitness[best_index]

    # Print progress
    if (iteration + 1) % 10 == 0:

        print(
            f"Iteration {iteration + 1}: "
            f"Best Throughput = "
            f"{global_best_fitness:.4f} Mbps"
        )


# ============================================================
# 9. Final Result
# ============================================================

optimal_power = global_best[:N]

optimal_bandwidth = global_best[N:] / 1e6


print("\n" + "=" * 50)
print("FINAL PSO RESULT")
print("=" * 50)

print(f"\nMaximum Network Throughput: "
      f"{global_best_fitness:.4f} Mbps")

print("\nOptimal Resource Allocation:")

for i in range(N):

    print(f"\nUser {i + 1}")
    print(f"  Channel Gain      : {channel_gain[i]}")
    print(f"  Power Allocation  : {optimal_power[i]:.4f} W")
    print(f"  Bandwidth         : {optimal_bandwidth[i]:.4f} MHz")


print("\n" + "-" * 50)

print(f"Total Power     : "
      f"{np.sum(optimal_power):.4f} W")

print(f"Total Bandwidth : "
      f"{np.sum(optimal_bandwidth):.4f} MHz")

print("-" * 50)

Iteration 10: Best Throughput = 99.6723 Mbps
Iteration 20: Best Throughput = 99.6723 Mbps
Iteration 30: Best Throughput = 99.6723 Mbps
Iteration 40: Best Throughput = 99.6723 Mbps
Iteration 50: Best Throughput = 99.6723 Mbps
Iteration 60: Best Throughput = 99.6723 Mbps
Iteration 70: Best Throughput = 99.6723 Mbps
Iteration 80: Best Throughput = 99.6723 Mbps
Iteration 90: Best Throughput = 99.6723 Mbps
Iteration 100: Best Throughput = 99.6723 Mbps

FINAL PSO RESULT

Maximum Network Throughput: 99.6723 Mbps

Optimal Resource Allocation:

User 1
  Channel Gain      : 1.0
  Power Allocation  : 10.0000 W
  Bandwidth         : 10.0000 MHz

User 2
  Channel Gain      : 0.8
  Power Allocation  : 0.0000 W
  Bandwidth         : 0.0000 MHz

User 3
  Channel Gain      : 0.5
  Power Allocation  : 0.0000 W
  Bandwidth         : 0.0000 MHz

--------------------------------------------------
Total Power     : 10.0000 W
Total Bandwidth : 10.0000 MHz
--------------------------------------------------
